In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
import json
from evaluation_utils import llm_structured_retry

from pydantic import BaseModel

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

In [6]:
documents_3 = []
for document in documents:
    if document['filename'] in ['01-agentic-rag/lessons/01-intro.md', '01-agentic-rag/lessons/02-environment.md', '01-agentic-rag/lessons/03-rag.md']:
        documents_3.append(document)

ground_truth_3 = []
usages_3 = []

for document in documents_3:
    records, usage = generate_ground_truth(document)
    ground_truth_3.extend(records)
    usages_3.append(usage)


In [7]:
# Q1. Generating questions
input_tokens_total = 0
for usage in usages_3:
    input_tokens_total += usage.input_tokens

input_tokens_total / len(usages_3)

1353.0

In [8]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(chunks)

In [11]:
from tqdm.auto import tqdm
import numpy as np
from embedder import Embedder

embed = Embedder()

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_vectors = embed.encode_batch([item['content'] for item in batch])
    X.extend(batch_vectors)

X = np.array(X)

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

  0%|          | 0/6 [00:00<?, ?it/s]

In [12]:
def text_search(query, num_results=5):
      return index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
      query_vector = embed.encode(query)
      return vindex.search(query_vector, num_results=num_results)


In [13]:
# Q2. First result with text search
q = ground_truth[0]["question"]
text_results = text_search(q)

print(q)
print(text_results[0]["filename"])


What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/03-rag.md


In [14]:
# Q3. First result with vector search
q = ground_truth[0]["question"]
vector_results = vector_search(q)

print(q)
print(vector_results[0]["filename"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/01-intro.md


In [15]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    expected_filename = q["filename"]
    results = search_function(q["question"])

    return [
        int(result["filename"] == expected_filename)
        for result in results
    ]


def compute_relevance_total(ground_truth, search_function):
    return [
        compute_relevance(q, search_function)
        for q in tqdm(ground_truth)
    ]


def hit_rate(relevance):
    hits = sum(1 in line for line in relevance)
    return hits / len(relevance)


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank, is_relevant in enumerate(line):
            if is_relevant:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance = compute_relevance_total(
        ground_truth,
        search_function,
    )

    return {
        "hit_rate": hit_rate(relevance),
        "mrr": mrr(relevance),
    }


In [17]:
# Q4. Evaluating text search
text_metrics = evaluate(ground_truth, text_search)
text_metrics['hit_rate']

  0%|          | 0/360 [00:00<?, ?it/s]

0.7583333333333333

In [18]:
# Q5. Evaluating vector search
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics['mrr']


  0%|          | 0/360 [00:00<?, ?it/s]

0.5486111111111112

In [19]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=5,
    )

In [21]:
# Q6. Tuning hybrid search
hybrid_metrics = {}

for k in [1, 50, 100, 200]:
    search_function = lambda query, k=k: hybrid_search(query, k=k)
    hybrid_metrics[k] = evaluate(ground_truth, search_function)

for k, metrics in hybrid_metrics.items():
    print(f"k={k}: MRR={metrics['mrr']:.4f}")

best_k = max(
    hybrid_metrics,
    key=lambda k: hybrid_metrics[k]["mrr"],
)

best_k


  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: MRR=0.6482
k=50: MRR=0.6379
k=100: MRR=0.6379
k=200: MRR=0.6379


1